**This notebook is the factory, handling handshake with Confluence, the deep crawl, cleaning, and final push to the vector DB.**

# 🏗️ Pillar A: Knowledge Base Ingestion Pipeline
**Project:** Yuan-Yuan's AI Knowledge Base  
**System Role:** Data Engineering & Vectorization Engine

### 🎯 Objective
This notebook is the "Librarian" of the system serving as the 'Ingestion Engine'. It automates the discovery of curated tribal knowledge from Yuan-Yuan's Confluence favorites. It cleans unstructured HTML and transforms it into a searchable, high-dimensional vector space.

### 🛠️ Key Architectural Features:
1. **Recursive Discovery:** Uses a custom Stage 2 Crawler to identify and follow technical links within documentation pages of interest.
2. **Context-Aware Cleaning:** Employs a BeautifulSoup-based parser to strip Confluence UI noise while preserving technical code blocks and lists.
3. **Idempotent Upsert:** Designed to be run repeatedly and for operational efficiency; it updates existing records in ChromaDB without creating duplicate embeddings.
4. **Vector Embeddings:** Utilizes the `all-MiniLM-L6-v2` Sentence Transformer to map text to a 384-dimensional semantic space.

---
*Status: Run this notebook to sync the Vector DB with your latest Confluence favorites.*</br>
*indempotent: a system designed as that running multiple time with the same input produces the exact same results as running it once*

In [ ]:
import pysqlite3
import sys
import os
import re
import requests
from requests.auth import HTTPBasicAuth

# Crucial for ChromaDB on Linux/SAS environments
sys.modules["sqlite3"] = pysqlite3

import chromadb
import rag_util

Loading Embedding Model...


In [6]:
# Try reloading the module to get latest changes
import importlib
importlib.reload(rag_util)
print("\nAfter reload - Contents of rag_util:", dir(rag_util))

Loading Embedding Model...

After reload - Contents of rag_util: ['BASE_URL', 'BeautifulSoup', 'HTTPBasicAuth', 'PASSWORD', 'SentenceTransformer', 'USERNAME', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'chunk_text', 'clean_confluence_html', 'create_embeddings', 'extract_confluence_links', 'get_child_pages', 'get_child_pages_EXPANDED', 'get_page_by_id', 'get_page_by_title', 'json', 'model', 'os', 'process_and_clean', 'process_with_crawler', 'requests', 'stage_2_crawler', 'step_3_pulling_actual_content']


In [ ]:

def run_full_sync():
    print("🔄 Starting Integrated Discovery (Favorites + Sub-pages + Stage 2 Crawler)...")
    
    # 1. Get initial favorited pages using rag_util
    favorites = rag_util.step_3_pulling_actual_content() 
    
    full_kb = []
    processed_ids = set()

    # --- Phase A: Process Favorites and their Children ---
    for fav in favorites:
        if fav['id'] not in processed_ids:
            full_kb.append(fav)
            processed_ids.add(fav['id'])
        
        # Discover and add direct children (DailyBio, Weekly Update, etc.)
        # children = rag_util.get_child_pages(fav['id'])
        children_data= rag_util.get_child_pages_EXPANDED(fav['id'])
        for child in children_data:
            # child_id = child.get('id')
            # if child_id and child_id not in processed_ids:
            if child['id'] not in processed_ids:
                print(f"   ✨ Found Child Page: {child['title']}")
                #child_data = rag_util.get_page_by_id(child_id)
                #if child_data:
                full_kb.append(child)
                processed_ids.add(child['id'])

    # --- Phase B: Stage 2 Crawler (Specific to Retirement Home) ---
    # Look for the 'PC SAS Retirement Home' page specifically to extract its internal links
    retirement_home = next(
        (p for p in full_kb if "PC SAS Retirement Home" in p['title'].replace('\xa0', ' ')), 
        None
    )

    if retirement_home:
        print(f"\n🕵️ Stage 2 Crawler: Extracting links from {retirement_home['title']}...")
        discovered_links = rag_util.extract_confluence_links(retirement_home['html'])
        
        for link in discovered_links:
            url = link['url'] 
            # Regex Pattern 1: Look for numeric page IDs
            id_match = re.search(r'(?:pages/|pageId=)(\d+)', url)
            # Regex Pattern 2: Look for Friendly Titles (/display/SPACE/Title)
            display_match = re.search(r'/display/([^/]+)/([^#?]+)', url)
            
            page_data = None
            if id_match:
                page_id = id_match.group(1)
                if page_id not in processed_ids:
                    print(f"   📥 Fetching Linked Page: {link['label']} (ID: {page_id})")
                    page_data = rag_util.get_page_by_id(page_id)
            elif display_match:
                space_key = display_match.group(1)
                title = display_match.group(2).replace('+', ' ')
                if title not in [p['title'] for p in full_kb]:
                    print(f"   📥 Fetching by Title: {title} (Space: {space_key})")
                    page_data = rag_util.get_page_by_title(space_key, title)
                    
            if page_data:
                full_kb.append(page_data)
                processed_ids.add(page_data['id'])

    # --- Phase C: Cleaning, Chunking, and Vectorizing ---
    clean_data = rag_util.process_and_clean(full_kb)
    all_chunks = []
    for page in clean_data:
        chunks = rag_util.chunk_text(page['clean_content'])
        for c in chunks:
            all_chunks.append({
                "source_id": page['id'],
                "source_title": page['title'],
                "chunk_text": c
            })
            
    # 2. Vectorize
    final_data = rag_util.create_embeddings(all_chunks)
    
    # 3. Push to ChromaDB (Persistent)
    client = chromadb.PersistentClient(path="./chroma_data")
    # Using get_or_create to ensure we don't crash if the collection exists
    collection = client.get_or_create_collection(name="document_store")
    
    collection.upsert(
        ids=[f"id_{i}" for i in range(len(final_data))],
        documents=[item['chunk_text'] for item in final_data],
        embeddings=[item['vector'] for item in final_data],
        metadatas=[{"source_id": i['source_id'], "source_title": i['source_title']} for i in final_data]
    )
    print(f"✅ Sync Complete. {len(final_data)} chunks indexed (Target: 56).")

# Run the full process
run_full_sync()

🔄 Starting Integrated Discovery (Favorites + Sub-pages + Stage 2 Crawler)...
Step 3: Pulling actual content for RAG knowledge base...
✅ Added to queue: Connect and Aim Ahead Home Page
✅ Added to queue: PC SAS Retirement Home
✅ Added to queue: Python Set Up Guide
   ✨ Found Child Page: DailyBio
   ✨ Found Child Page: Weekly Update Process
   ✨ Found Child Page: Python Virtual Environment Set Up:

🕵️ Stage 2 Crawler: Extracting links from PC SAS Retirement Home...
   📥 Fetching by Title: Work with SAS Datasets and Programs without a PC SAS Installation (Space: EIM)
   📥 Fetching Linked Page: Connecting Power BI to SAS Datasets - EIM - Confluence (ID: 549161132)
   📥 Fetching Linked Page: Working with Large SAS Datasets - RDBMS - Confluence (ID: 562955018)

Step 3.1: Cleaning 9 pages...
--- CLEANED: Connect and Aim Ahead Home Page ---
Connect
NCI Master Data Dictionary Link
:
https://github.com/Analyticsphere/ConnectMasterAndSurveyCo...
--- CLEANED: DailyBio ---
Daily Bio file correction: